In [6]:
!nvidia-smi -L || true
%pip -q install -U unsloth "transformers>=4.42.0" "accelerate>=0.31.0" \
  bitsandbytes peft datasets trl pymupdf tiktoken openai huggingface_hub

import torch, platform
print("Python:", platform.python_version())
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| CC:", torch.cuda.get_device_capability(0))


GPU 0: Tesla T4 (UUID: GPU-d2314ee6-b916-dc76-d8ad-9bd64d281a5f)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 946.9/946.9 kB 20.6 MB/s eta 0:00:00
Python: 3.12.11
Torch: 2.8.0+cu126 | CUDA: True
GPU: Tesla T4 | CC: (7, 5)


## Mount Drive + project paths

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/ai-agent"
else:
    BASE = "/content/ai-agent"

import pathlib, os
DATA_DIR      = f"{BASE}/data/arxiv"     # PDFs here
ARTIFACTS_DIR = f"{BASE}/artifacts"      # generated files here
SECRETS_DIR   = f"{BASE}/secrets"        # key files here
for p in [DATA_DIR, ARTIFACTS_DIR, SECRETS_DIR]:
    pathlib.Path(p).mkdir(parents=True, exist_ok=True)

print("BASE        :", BASE)
print("DATA_DIR    :", DATA_DIR)
print("ARTIFACTS   :", ARTIFACTS_DIR)
print("SECRETS_DIR :", SECRETS_DIR)


## Keys & tokens



In [9]:
import os

def _mask(token: str, left: int = 6, right: int = 4) -> str:
    token = token or ""
    if len(token) <= left + right:
        return "*" * len(token)
    return token[:left] + "…" + token[-right:]

def _from_colab(name: str) -> str:
    """Fetch a secret from Colab's Secrets (UI: left sidebar → Secrets)."""
    try:
        from google.colab import userdata  # type: ignore
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

# Read ONLY from Colab Secrets
OPENAI_API_KEY = _from_colab("openaikey")
HF_TOKEN       = _from_colab("huggingfacekey")

# Export to env for downstream libs
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if HF_TOKEN:
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

print("OpenAI key set:", bool(OPENAI_API_KEY), _mask(OPENAI_API_KEY) if OPENAI_API_KEY else "")
print("HF token set  :", bool(HF_TOKEN),       _mask(HF_TOKEN)       if HF_TOKEN else "")

# Optional: log in to Hugging Face if token present
if HF_TOKEN:
    try:
        from huggingface_hub import login, whoami
        login(token=HF_TOKEN)
        user = whoami()
        print("HF user:", user.get("name") or user.get("email") or user.get("id"))
    except Exception as e:
        print("Hugging Face login failed:", e)


OpenAI key set: True sk-pro…_mcA
HF token set  : True hf_bDa…EhQq
HF user: CourierCat


## Picking a model

In [11]:
from huggingface_hub import hf_hub_download

CANDIDATES = [
    "unsloth/Qwen2-7B-Instruct-bnb-4bit",
]

BASE_MODEL = None
for mid in CANDIDATES:
    try:
        hf_hub_download(mid, "config.json", token=HF_TOKEN or None)
        BASE_MODEL = mid
        print("✅ Using base model:", BASE_MODEL)
        break
    except Exception as e:
        print(f"Skipping {mid}: {e}")
assert BASE_MODEL, "No accessible base model found. Provide HF token for Llama-3 or use an open model."

MAX_SEQ_LEN = 4096


✅ Using base model: unsloth/Qwen2-7B-Instruct-bnb-4bit


## Load 4-bit model

In [13]:
from unsloth import FastLanguageModel

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

wanted = set(TARGET_MODULES)
found = set()
for name, _mod in model.named_modules():
    last = name.split(".")[-1]
    if last in wanted:
        found.add(last)
print("LoRA will target:", sorted(found))
missing = wanted - found
if missing:
    print("Not found on this model (you can ignore if empty):", sorted(missing))

# Attach LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=TARGET_MODULES,
    use_rslora=True,
    loftq_config=None,
)

print("LoRA adapters injected.")


LoRA will target: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']


Unsloth 2025.9.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA adapters injected.


## Loading dataset

In [18]:
from pathlib import Path
import fitz, json
from datasets import Dataset

pdfs = sorted(Path("data/arxiv").glob("*.pdf"))
assert pdfs, f"No PDFs found in {DATA_DIR}. Upload some PDFs first."

def extract_text(pdf_path: Path) -> str:
    with fitz.open(pdf_path) as doc:
        return "\n".join(p.get_text("text") for p in doc)

records = []
for fp in pdfs:
    try:
        txt = extract_text(fp)
        if txt.strip():
            records.append({"source": fp.name, "text": txt})
        else:
            print("Empty extract:", fp.name)
    except Exception as e:
        print("Failed:", fp.name, e)

corpus_ds = Dataset.from_list(records)
print(corpus_ds)
print("Docs:", len(corpus_ds))

Path(ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)
with open(f"{ARTIFACTS_DIR}/corpus.jsonl", "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Saved:", f"{ARTIFACTS_DIR}/corpus.jsonl")


Dataset({
    features: ['source', 'text'],
    num_rows: 50
})
Docs: 50
Saved: /content/drive/MyDrive/ai-agent/artifacts/corpus.jsonl


## Seeds & corpus stats

In [19]:
import random, numpy as np, torch, math

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

avg_chars = sum(len(r["text"]) for r in corpus_ds) / max(1, len(corpus_ds))
print(f"Corpus docs: {len(corpus_ds)} | Avg chars/doc: {int(avg_chars):,}")


Corpus docs: 50 | Avg chars/doc: 67,301


## Chunking utils

In [20]:
import re
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
except Exception:
    enc = None

def tok_len(s: str) -> int:
    return len(enc.encode(s)) if enc else len(s)

def chunk_text(text: str, max_tokens=400, overlap=60):
    paras = re.split(r"\n\s*\n", text)
    chunks, buf, buf_toks = [], [], 0
    for p in paras:
        t = tok_len(p)
        if buf_toks + t > max_tokens and buf:
            joined = "\n\n".join(buf)
            chunks.append(joined)
            if overlap > 0:
                if enc:
                    tail = enc.decode(enc.encode(joined)[-overlap:])
                else:
                    tail = joined[-overlap:]
                buf, buf_toks = [tail], tok_len(tail)
            else:
                buf, buf_toks = [], 0
        buf.append(p); buf_toks += t
    if buf:
        chunks.append("\n\n".join(buf))
    return chunks

def build_contexts(ds, max_per_doc=6):
    rows = []
    for row in ds:
        for c in chunk_text(row["text"]):
            rows.append({"source": row["source"], "context": c})
            if len(rows) >= max_per_doc * len(ds):
                break
    return rows

contexts = build_contexts(corpus_ds, max_per_doc=6)
print("Contexts:", len(contexts))


Contexts: 334


## Synthetic Q&A generator

In [23]:
from openai import OpenAI
import json, os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

SYS = (
"Generate ONE high-quality question and answer strictly about the given context. "
"Return VALID JSON with keys: question, answer. Keep answers grounded; if unclear, note limitations."
)

def gen_qa(context: str) -> dict:
    prompt = f"Context:\n{context}\n\nReturn JSON: {{\"question\":..., \"answer\":...}}"
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"system","content":SYS},
                  {"role":"user","content":prompt}],
        temperature=0.2,
    )
    raw = resp.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = "\n".join([ln for ln in raw.splitlines() if not ln.strip().startswith("```")]).strip()
    try:
        obj = json.loads(raw)
        if not isinstance(obj, dict) or "question" not in obj or "answer" not in obj:
            raise ValueError("missing keys")
        return obj
    except Exception:
        return {"question":"(invalid)", "answer": raw}

res = gen_qa(contexts[0]["context"])
print("Q:", res.get("question", "<missing>"))
print("A:", res.get("answer", "<missing>"))


Q: What are the main contributions of the proposed optimization framework in the context of 3D Gaussian splatting?
A: The proposed optimization framework integrates multi-sample anti-aliasing (MSAA) with dual geometric constraints to improve detail preservation in 3D Gaussian splatting. It introduces an adaptive weighting strategy that focuses on under-reconstructed regions through dynamic gradient analysis, and gradient differential constraints that enforce geometric regularization at object boundaries. This targeted approach allows for better allocation of computational resources to critical areas, resulting in reduced aliasing artifacts and improved rendering of high-frequency textures and sharp discontinuities, while maintaining real-time efficiency.


## Generate N QAs

In [ ]:
import random, itertools, json, pathlib
N = 50
random.seed(SEED)

picked = contexts if len(contexts) >= N else list(itertools.islice(itertools.cycle(contexts), N))
picked = random.sample(picked, min(N, len(picked)))

out_path = f"{ARTIFACTS_DIR}/synthetic_qa.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for i, ctx in enumerate(picked, 1):
        qa = gen_qa(ctx["context"])
        text = (
            "<|system|>You are an academic Q&A assistant.<|end|>"
            f"<|user|>{qa['question']}<|end|>"
            f"<|assistant|>{qa['answer']}<|end|>"
        )
        f.write(json.dumps({"text": text, "source": ctx["source"]}, ensure_ascii=False) + "\n")
        if i % 20 == 0: print(f"{i}/{len(picked)}…")
print("Saved:", out_path)
# note: cell shows failure because I've been rate limited by openai, the first pass still ran with only N=40, so that result will be shown here

## Loading dataset and making train/val split

In [ ]:
from datasets import load_dataset, DatasetDict
import glob

files = sorted(glob.glob(f"{ARTIFACTS_DIR}/synthetic_qa*.jsonl"))
assert files, f"No synthetic QA found in {ARTIFACTS_DIR}"

ds_all = load_dataset("json", data_files=files, split="train")
ds = ds_all.train_test_split(test_size=0.1, seed=SEED)
ds_train, ds_val = ds["train"], ds["test"]
print("Train size:", len(ds_train), "| Val size:", len(ds_val))
print(ds_train[:2])


## Fine tune with QLora

In [32]:
from trl import SFTTrainer
import transformers, torch, math

cc_major = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
bf16_ok = torch.cuda.is_available() and cc_major >= 8
fp16_ok = torch.cuda.is_available() and not bf16_ok

OUTPUT_DIR = f"{ARTIFACTS_DIR}/finetune_out"

args = transformers.TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    bf16=bf16_ok,
    fp16=fp16_ok,
    logging_steps=10,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
    args=args,
)

train_out = trainer.train()
metrics = trainer.evaluate()
print("Eval metrics:", metrics)
if "eval_loss" in metrics:
    try:
        print("Perplexity:", math.exp(metrics["eval_loss"]))
    except OverflowError:
        pass

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved adapters/tokenizer to:", OUTPUT_DIR)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/36 [00:00<?, ? examples/s]

num_proc must be <= 5. Reducing num_proc to 5 for dataset of size 5.


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/5 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 36 | Num Epochs = 1 | Total steps = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Eval metrics: {'eval_loss': 1.530741572380066, 'eval_runtime': 1.1435, 'eval_samples_per_second': 4.372, 'eval_steps_per_second': 0.874, 'epoch': 1.0}
Perplexity: 4.6216028048274636
✅ Saved adapters/tokenizer to: /content/drive/MyDrive/ai-agent/artifacts/finetune_out


## Qualitative eval after tuning

In [33]:
import torch, json

def chat(q, max_new_tokens=200):
    msgs = [{"role":"system","content":"You are a concise academic Q&A assistant."},
            {"role":"user","content":q}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True).split(q)[-1].strip()

eval_qs = [
    "List two datasets commonly used in LLM benchmarks.",
    "What datasets recur in multimodal papers in this corpus?",
    "Name one frequent limitation discussed across safety papers.",
    "Summarize a common methodology trend you saw.",
    "Suggest one realistic future direction for LLM research.",
    "What benchmarks are often used for instruction following?",
    "Which evaluation pitfalls are mentioned repeatedly?"
]

outs = [{"q": q, "a": chat(q)} for q in eval_qs]
with open(f"{ARTIFACTS_DIR}/post_tune_eval.json","w",encoding="utf-8") as f:
    json.dump(outs, f, indent=2, ensure_ascii=False)

outs


[{'q': 'List two datasets commonly used in LLM benchmarks.',
  'a': "assistant\nTwo commonly used datasets for evaluating Large Language Models (LLMs) include:\n\n1. **The Pile**: This is a large, diverse dataset of internet content that aims to be representative of the kind of text found on the web today. It's designed to help researchers train and evaluate models on a wide range of topics and styles, making it suitable for assessing general language understanding and generation abilities.\n\n2. **Wikitext**: This dataset consists of Wikipedia articles, which are known for their structured nature and high-quality content. It's particularly useful for evaluating language models' ability to generate coherent paragraphs and sentences, as well as their understanding of context and semantics within the structured information found in Wikipedia pages.\n\nThese datasets are crucial because they provide a benchmark for comparing different LLMs, helping researchers understand strengths and wea

## Stand alone inference

In [34]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

BASE_ID = BASE_MODEL
ADAPTER_DIR = f"{ARTIFACTS_DIR}/finetune_out"

base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name=BASE_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
base_tok.pad_token = base_tok.eos_token

def chat_infer(q, max_new_tokens=200):
    msgs = [{"role":"system","content":"You are a concise academic Q&A assistant."},
            {"role":"user","content":q}]
    prompt = base_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = base_tok(prompt, return_tensors="pt").to(ft_model.device)
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return base_tok.decode(out[0], skip_special_tokens=True).split(q)[-1].strip()

print("→", chat_infer("Give two datasets frequently used in the papers."))


==((====))==  Unsloth 2025.9.4: Fast Qwen2 patching. Transformers: 4.56.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
→ assistant
In academic research, various datasets are commonly used depending on the field of study. Here are two examples:

1. **MNIST Dataset**: This is one of the most widely used datasets for training and testing in computer vision tasks, particularly for handwritten digit recognition. It contains 60,000 training images and 10,000 test images of 28x28 grayscale images of the ten digits from 0 to 9.

2. **IMDB Reviews Dataset**: This dataset is often used in natural language processing tasks, specifically sentiment analysis. It consis

## Documentation

In [ ]:
import json, pathlib
card = {
  "name": "synthetic_qa_week7",
  "schema": {"text": "string", "source": "string"},
  "count": int(len(ds_all)) if 'ds_all' in globals() else None,
  "source_docs": int(len(corpus_ds)) if 'corpus_ds' in globals() else None,
  "license": "CC-BY-4.0 (corpus PDFs not redistributed)",
  "generation_model": "gpt-4o-mini",
  "base_model": BASE_MODEL,
  "notes": "Synthetic Q&A generated from local PDF corpus; trained with QLoRA on 4-bit base."
}
path = f"{ARTIFACTS_DIR}/dataset_card.json"
pathlib.Path(path).write_text(json.dumps(card, indent=2))
print("✅ Wrote dataset card:", path)
